In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, ArrayType, MapType,
                               StringType, LongType, DoubleType)
from pyspark.sql.window import Window

BRONZE = "/Volumes/workspace/bronze/raw_files/places"
TABELA = "workspace.silver.places"
REJEITADOS = "workspace.silver.places_rejected"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA} (
  place_id STRING,
  tipo_local STRING,
  subtipo STRING,
  name STRING,
  lat DOUBLE,
  lon DOUBLE,
  ingestion_date DATE,
  source_file STRING,
  updated_at TIMESTAMP
)
""")

schema = StructType([StructField("elements", ArrayType(StructType([
    StructField("type", StringType()),
    StructField("id", LongType()),
    StructField("lat", DoubleType()),
    StructField("lon", DoubleType()),
    StructField("tags", MapType(StringType(), StringType())),
])))])

raw = (spark.read.schema(schema).option("multiLine", True).json(BRONZE)
       .withColumn("source_file", F.col("_metadata.file_path")))

tags = F.col("e.tags")

df = (raw.select("source_file", F.explode("elements").alias("e"))
      .select(
          F.concat_ws("/", F.col("e.type"), F.col("e.id").cast("string")).alias("place_id"),
          F.trim(tags["name"]).alias("name"),
          tags["place"].alias("place"),
          tags["railway"].alias("railway"),
          tags["station"].alias("station_tag"),
          tags["subway"].alias("subway_tag"),
          tags["train"].alias("train_tag"),
          F.col("e.lat").alias("lat"),
          F.col("e.lon").alias("lon"),
          F.to_date(F.regexp_extract("source_file", r"ingestion_date=(\d{4}-\d{2}-\d{2})", 1)).alias("ingestion_date"),
          "source_file",
      ))

# Se o mesmo local veio em mais de um dia, fica com o mais recente
w = (Window.partitionBy("place_id")
     .orderBy(F.col("ingestion_date").desc(), F.col("source_file").desc()))
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

print("Locais únicos:", df.count())

In [0]:
df = (df
      .withColumn("tipo_local",
                  F.when(F.col("place").isNotNull(), "bairro")
                   .when(F.col("railway") == "station", "estacao"))
      .withColumn("subtipo",
                  F.when(F.col("place").isNotNull(), F.col("place"))
                   .when((F.col("station_tag") == "subway") | (F.col("subway_tag") == "yes"), "metro")
                   .when((F.col("station_tag") == "train") | (F.col("train_tag") == "yes"), "trem")
                   .otherwise("estacao_outra")))

display(df.groupBy("tipo_local", "subtipo").count().orderBy("tipo_local", F.desc("count")))

In [0]:
motivo = (F.when(F.col("tipo_local").isNull(), "tipo_desconhecido")
           .when(F.col("name").isNull() | (F.col("name") == ""), "sem_nome")
           .when(F.col("lat").isNull() | F.col("lon").isNull(), "sem_coordenada")
           .when(~(F.col("lat").between(-24.1, -23.0) & F.col("lon").between(-47.0, -46.0)), "fora_de_sp")
           .otherwise(None))

df = df.withColumn("motivo_rejeicao", motivo)

colunas = ["place_id", "tipo_local", "subtipo", "name", "lat", "lon", "ingestion_date", "source_file"]

validos = (df.filter("motivo_rejeicao IS NULL").select(*colunas)
             .withColumn("updated_at", F.current_timestamp()))
rejeitados = (df.filter("motivo_rejeicao IS NOT NULL").select(*colunas, "motivo_rejeicao")
                .withColumn("rejected_at", F.current_timestamp()))

print("Válidos:", validos.count(), "| Rejeitados:", rejeitados.count())
display(rejeitados.groupBy("motivo_rejeicao").count())

(rejeitados.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REJEITADOS))

validos.createOrReplaceTempView("places_novo")

spark.sql(f"""
MERGE INTO {TABELA} AS t
USING places_novo AS s
ON t.place_id = s.place_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
display(spark.sql(f"""
SELECT tipo_local, subtipo, COUNT(*) AS total
FROM {TABELA}
GROUP BY tipo_local, subtipo
ORDER BY tipo_local, total DESC
"""))

display(spark.sql(f"""
SELECT tipo_local, subtipo, name, lat, lon
FROM {TABELA}
WHERE lower(name) LIKE '%sacom%'
"""))